# Mutual Fund Data Cleaning & Star Schema Loading

This notebook implements the data cleaning and validation routines for the Bluestock Mutual Fund Analytics Capstone, followed by loading the datasets into a SQLite star schema database.

### Objectives:
1. **Clean `nav_history.csv`**: Parse dates, sort, forward-fill missing NAVs for weekends/holidays, remove duplicates, validate `NAV > 0`.
2. **Clean `investor_transactions.csv`**: Standardize transaction types, validate `amount > 0`, fix date formats, validate KYC status.
3. **Clean `scheme_performance.csv`**: Validate numeric return values, flag mathematical anomalies, validate expense ratio ranges (0.1% - 2.5%).
4. **Load into SQLite Star Schema**: Populate database tables with referential integrity constraints.

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, text

project_root = Path('..')
db_path = project_root / 'data' / 'db' / 'bluestock_mf.db'
processed_dir = project_root / 'data' / 'processed'
raw_dir = project_root / 'data' / 'raw'

## 1. Inspect Cleaning Logic & Results
Let's look at samples of the cleaned datasets exported by our cleaning modules.

In [ ]:
# Load a sample of cleaned NAV history
df_nav = pd.read_csv(processed_dir / '02_nav_history.csv')
print(f"Cleaned NAV History Shape: {df_nav.shape}")
df_nav.head(5)

In [ ]:
# Load a sample of cleaned investor transactions
df_tx = pd.read_csv(processed_dir / '08_investor_transactions.csv')
print(f"Cleaned Investor Transactions Shape: {df_tx.shape}")
df_tx.head(5)

In [ ]:
# Load a sample of cleaned scheme performance
df_perf = pd.read_csv(processed_dir / '07_scheme_performance.csv')
print(f"Cleaned Scheme Performance Shape: {df_perf.shape}")
print(f"Anomalous schemes flagged: {df_perf['is_anomalous'].sum()}")
df_perf.head(5)

## 2. Verify Schema Loading & Integrity Checks
We connect to the new SQLite database and verify the tables and row counts loaded.

In [ ]:
# Connect to database
conn = sqlite3.connect(db_path)

# Query loaded tables
tables_df = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in Database:")
print(tables_df)

In [ ]:
# Check row counts for fact and dimension tables
row_counts = {}
for t in tables_df['name']:
    if t == 'sqlite_sequence': continue
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {t}", conn).iloc[0]['count']
    row_counts[t] = count

print("Database Row Counts:")
for tbl, cnt in row_counts.items():
    print(f"  {tbl:<22}: {cnt} rows")
conn.close()

## 3. Findings
1. **NAV History Cleaning**: Reindexed the dates to cover full daily calendar date ranges, and forward-filled (`ffill`) missing NAV for weekends and market holidays. NAV values were validated to be strictly greater than 0.
2. **Transaction Cleaning**: Standardized the transaction type field to `SIP`, `Lumpsum`, or `Redemption`. KYC status was normalized to the enum values `Verified` or `Pending`.
3. **Performance Anomalies**: Found that standard deviation and returns are successfully loaded. Liquid funds with extremely low std deviation were flagged as `is_anomalous`.